# PhenoBench Multiclass — 3×3 tiled raw RGB images

Builds a Kaggle dataset of **lossless tiled RGB images** that matches the
`*_annotations.json` files created by the paired
[**PhenoBench Multiclass Export — 3×3 tiled (512)**](https://www.kaggle.com/code/freimutdiener/export-mc-phenobench-tiled-no-partials) notebook.

Attach both inputs before running:

1. `phenobench-raw-dataset-v1-1-0`
2. The output dataset of the TFRecord / COCO export notebook containing
   `train_annotations.json`, `val_annotations.json`, `test_annotations.json`,
   `true_eval_annotations.json`, `dataset_metadata.json`, and related artifacts.

The output layout is:

```text
phenobench_multiclass_3x3_tiled_512_raw/
├── images/
│   ├── <source>_tile0.png
│   ├── ...
│   └── <source>_tile8.png
├── annotations/
│   ├── train_annotations.json
│   ├── val_annotations.json
│   ├── test_annotations.json
│   ├── true_eval_annotations.json
│   ├── val_test_split.json
│   ├── rep_dataset.json
│   ├── label_map.pbtxt
│   └── dataset_metadata.json
└── raw_images_metadata.json
```

The annotation JSON files use bare file names, so consumers should use
`images/` as their image root.

> **Important:** this intentionally reuses `TiledPhenoBench`, rather than
> reimplementing tile coordinates. With a 3×3 grid and 0.5 overlap, the corner
> tiles are 512×512 but the middle row/column tiles are larger (typically
> 683 px); that is exactly what the preceding annotation exporter used.


## Environment

Use the exact same `agri-vision-edge` commit as the annotation export notebook.
The notebook only writes RGB tiles; it does not recreate labels.


In [1]:
!python --version
!pip install -q --no-cache-dir phenobench
!pip install -q --no-deps \
  git+https://github.com/frdiener/agri-vision-edge.git@abf74003b804d5c7130e368e0ae0eef696fa047e

Python 3.10.10


## Configuration

In [2]:
from __future__ import annotations

import json
import shutil
from collections import Counter
from pathlib import Path

import numpy as np
from PIL import Image
from tqdm.auto import tqdm

from phenobench import PhenoBench

from agri_vision_edge.data.tiling import FilterConfig, TiledPhenoBench


# Must match the paired export notebook.
ROWS = 3
COLS = 3
OVERLAP = 0.5
PARTIAL_THRESHOLD = 0.5

RAW_DATASET_ROOT = Path(
    "/kaggle/input/datasets/freimutdiener/"
    "phenobench-raw-dataset-v1-1-0/PhenoBench"
)

# Change this only to the Kaggle input directory containing the artifacts
# generated by the paired TFRecord / COCO export notebook.
ANNOTATIONS_ROOT = Path(
    "/kaggle/input/datasets/freimutdiener/"
    "mc-phenobench-tiled-no-partials"
)

OUTPUT_ROOT = Path(
    "/kaggle/working/phenobench_multiclass_3x3_tiled_512_raw"
)
IMAGES_ROOT = OUTPUT_ROOT / "images"
COPIED_ARTIFACTS_ROOT = OUTPUT_ROOT / "annotations"

assert RAW_DATASET_ROOT.exists(), RAW_DATASET_ROOT
assert ANNOTATIONS_ROOT.exists(), ANNOTATIONS_ROOT

# The paired exporter needs these labels to regenerate tile-local boxes.
# They have no effect on the saved RGB pixel data, but retaining them makes
# the image wrapper exactly identical to the annotation-export wrapper.
FILTER_CONFIG = FilterConfig(
    min_instance_pixels=32,
    min_bbox_width=4,
    min_bbox_height=4,
    min_bbox_area=32,
)

ANNOTATION_ARTIFACTS = (
    "train_annotations.json",
    "true_eval_annotations.json",
    "val_annotations.json",
    "test_annotations.json",
    "label_map.pbtxt",
    "rep_dataset.json",
    "val_test_split.json",
    "dataset_metadata.json",
)

missing = [name for name in ANNOTATION_ARTIFACTS if not (ANNOTATIONS_ROOT / name).is_file()]
assert not missing, (
    "The annotation input does not look like the output of the paired export "
    f"notebook. Missing: {missing}"
)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
IMAGES_ROOT.mkdir(parents=True, exist_ok=True)
COPIED_ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)

print("Raw input:       ", RAW_DATASET_ROOT)
print("Annotations input:", ANNOTATIONS_ROOT)
print("Output:          ", OUTPUT_ROOT)

/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


Raw input:        /kaggle/input/datasets/freimutdiener/phenobench-raw-dataset-v1-1-0/PhenoBench
Annotations input: /kaggle/input/datasets/freimutdiener/mc-phenobench-tiled-no-partials
Output:           /kaggle/working/phenobench_multiclass_3x3_tiled_512_raw


## Construct the identical tiled datasets

Do **not** substitute a manual `512×512` crop here. The wrapper determines the
same nine crops and names used by the exporter.


In [3]:
def load_raw_split(split: str) -> PhenoBench:
    return PhenoBench(
        root=RAW_DATASET_ROOT,
        split=split,
        target_types=[
            "semantics",
            "plant_instances",
            "plant_visibility",
        ],
        ignore_partial=False,
    )


def make_tiled(dataset: PhenoBench) -> TiledPhenoBench:
    return TiledPhenoBench(
        dataset,
        rows=ROWS,
        cols=COLS,
        overlap=OVERLAP,
        filter_config=FILTER_CONFIG,
        partial_threshold=PARTIAL_THRESHOLD,
    )


raw_train = load_raw_split("train")
raw_val = load_raw_split("val")

train_dataset = make_tiled(raw_train)
val_dataset = make_tiled(raw_val)

print(f"Train: {len(raw_train)} frames → {len(train_dataset)} tiles")
print(f"Val:   {len(raw_val)} frames → {len(val_dataset)} tiles")
print("Tile geometry (tile index → x0,y0,x1,y1):")
for i, tile in enumerate(train_dataset.tiles):
    print(f"  {i}: ({tile.x0}, {tile.y0}) → ({tile.x1}, {tile.y1}) "
          f"= {tile.width}×{tile.height}")

Train: 1407 frames → 12663 tiles
Val:   772 frames → 6948 tiles
Tile geometry (tile index → x0,y0,x1,y1):
  0: (0, 0) → (512, 512) = 512×512
  1: (256, 0) → (768, 512) = 512×512
  2: (512, 0) → (1024, 512) = 512×512
  3: (0, 256) → (512, 768) = 512×512
  4: (256, 256) → (768, 768) = 512×512
  5: (512, 256) → (1024, 768) = 512×512
  6: (0, 512) → (512, 1024) = 512×512
  7: (256, 512) → (768, 1024) = 512×512
  8: (512, 512) → (1024, 1024) = 512×512


## Write tiled RGB images

Saving through `tiled_dataset[index]["image"]` uses the wrapper's exact crop and
its exact `<stem>_tile<index><suffix>` name. Existing files are verified rather
than silently overwritten with potentially incompatible content.


In [4]:
def save_tiles(dataset: TiledPhenoBench, split_name: str) -> dict:
    written = 0
    reused = 0
    suffixes = Counter()

    for dataset_index in tqdm(
        range(len(dataset)),
        desc=f"Writing {split_name} tiles",
        unit="image",
    ):
        sample = dataset[dataset_index]
        image_name = Path(sample["image_name"]).name
        target = IMAGES_ROOT / image_name

        image = sample["image"]
        if image.mode != "RGB":
            image = image.convert("RGB")

        suffixes[target.suffix.lower()] += 1

        if target.exists():
            with Image.open(target) as existing:
                if existing.size != image.size:
                    raise RuntimeError(
                        f"Existing tile has a wrong geometry: {target} "
                        f"({existing.size} != {image.size})"
                    )
            reused += 1
            continue

        # PNG source names remain PNG; JPEG source names remain JPEG.
        # No resize and no lossy format conversion happens here.
        image.save(target)
        written += 1

    return {
        "split": split_name,
        "tiles": len(dataset),
        "written": written,
        "reused": reused,
        "suffixes": dict(sorted(suffixes.items())),
    }


train_write_stats = save_tiles(train_dataset, "train")
val_write_stats = save_tiles(val_dataset, "val")

print(train_write_stats)
print(val_write_stats)

Writing train tiles:   0%|          | 0/12663 [00:00<?, ?image/s]

Writing val tiles:   0%|          | 0/6948 [00:00<?, ?image/s]

{'split': 'train', 'tiles': 12663, 'written': 12663, 'reused': 0, 'suffixes': {'.png': 12663}}
{'split': 'val', 'tiles': 6948, 'written': 6948, 'reused': 0, 'suffixes': {'.png': 6948}}


## Copy the paired annotation artifacts

They are copied unchanged, so their image IDs, partial/do-not-care flags, split
indices, and representative-dataset indices retain the meaning established by
the export notebook.


In [5]:
for name in ANNOTATION_ARTIFACTS:
    source = ANNOTATIONS_ROOT / name
    target = COPIED_ARTIFACTS_ROOT / name
    shutil.copy2(source, target)

print("Copied:")
for path in sorted(COPIED_ARTIFACTS_ROOT.iterdir()):
    print(" -", path.name)

Copied:
 - dataset_metadata.json
 - label_map.pbtxt
 - rep_dataset.json
 - test_annotations.json
 - train_annotations.json
 - true_eval_annotations.json
 - val_annotations.json
 - val_test_split.json


## Compatibility verification

This checks every COCO `images[*].file_name` referenced by the copied JSON
against the image directory. It also checks pixel dimensions, which catches
wrong overlap geometry, wrong tile ordering, or accidental resize/re-encoding
errors.


In [6]:
def verify_coco_images(annotation_path: Path) -> dict:
    coco = json.loads(annotation_path.read_text())
    missing = []
    wrong_size = []
    duplicate_names = []

    names = [entry["file_name"] for entry in coco["images"]]
    counts = Counter(names)
    duplicate_names = sorted(name for name, count in counts.items() if count > 1)

    for entry in tqdm(
        coco["images"],
        desc=f"Checking {annotation_path.name}",
        unit="image",
    ):
        image_path = IMAGES_ROOT / entry["file_name"]

        if not image_path.is_file():
            missing.append(entry["file_name"])
            continue

        with Image.open(image_path) as image:
            actual_size = image.size

        expected_size = (int(entry["width"]), int(entry["height"]))
        if actual_size != expected_size:
            wrong_size.append(
                {
                    "file_name": entry["file_name"],
                    "expected": expected_size,
                    "actual": actual_size,
                }
            )

    assert not missing, (
        f"{annotation_path.name}: {len(missing)} referenced image(s) are missing; "
        f"first entries: {missing[:10]}"
    )
    assert not wrong_size, (
        f"{annotation_path.name}: {len(wrong_size)} image dimension mismatch(es); "
        f"first entries: {wrong_size[:3]}"
    )
    assert not duplicate_names, (
        f"{annotation_path.name}: duplicate image file names: "
        f"{duplicate_names[:10]}"
    )

    return {
        "annotation_file": annotation_path.name,
        "images": len(coco["images"]),
        "annotations": len(coco["annotations"]),
        "all_images_present": True,
        "all_dimensions_match": True,
    }


verification = [
    verify_coco_images(COPIED_ARTIFACTS_ROOT / name)
    for name in (
        "train_annotations.json",
        "true_eval_annotations.json",
        "val_annotations.json",
        "test_annotations.json",
    )
]

for result in verification:
    print(result)

Checking train_annotations.json:   0%|          | 0/12663 [00:00<?, ?image/s]

Checking true_eval_annotations.json:   0%|          | 0/6948 [00:00<?, ?image/s]

Checking val_annotations.json:   0%|          | 0/3474 [00:00<?, ?image/s]

Checking test_annotations.json:   0%|          | 0/3474 [00:00<?, ?image/s]

{'annotation_file': 'train_annotations.json', 'images': 12663, 'annotations': 47452, 'all_images_present': True, 'all_dimensions_match': True}
{'annotation_file': 'true_eval_annotations.json', 'images': 6948, 'annotations': 27306, 'all_images_present': True, 'all_dimensions_match': True}
{'annotation_file': 'val_annotations.json', 'images': 3474, 'annotations': 13740, 'all_images_present': True, 'all_dimensions_match': True}
{'annotation_file': 'test_annotations.json', 'images': 3474, 'annotations': 13566, 'all_images_present': True, 'all_dimensions_match': True}


## Dataset metadata and final check

In [7]:
image_paths = sorted(
    path for path in IMAGES_ROOT.iterdir()
    if path.is_file()
)

metadata = {
    "source_dataset": "PhenoBench v1.1.0 raw RGB images",
    "source_annotation_artifacts": str(ANNOTATIONS_ROOT),
    "image_root": "images",
    "annotation_root": "annotations",
    "tiling": {
        "rows": ROWS,
        "cols": COLS,
        "overlap": OVERLAP,
        "tile_order": "row-major, tile0 through tile8",
        "filename_pattern": "<source_stem>_tile<tile_index><source_suffix>",
    },
    "splits_written": {
        "train": train_write_stats,
        # val is deliberately complete because true_eval contains all original
        # validation tiles and val/test are subsets of it.
        "val_source": val_write_stats,
    },
    "images_written_total": len(image_paths),
    "coco_compatibility": verification,
}

(OUTPUT_ROOT / "raw_images_metadata.json").write_text(
    json.dumps(metadata, indent=2) + "\n"
)

assert image_paths, "No tiled images were written."
assert (OUTPUT_ROOT / "raw_images_metadata.json").is_file()

print(json.dumps(metadata, indent=2))
print("\nDataset ready:")
print(OUTPUT_ROOT)

{
  "source_dataset": "PhenoBench v1.1.0 raw RGB images",
  "source_annotation_artifacts": "/kaggle/input/datasets/freimutdiener/mc-phenobench-tiled-no-partials",
  "image_root": "images",
  "annotation_root": "annotations",
  "tiling": {
    "rows": 3,
    "cols": 3,
    "overlap": 0.5,
    "tile_order": "row-major, tile0 through tile8",
    "filename_pattern": "<source_stem>_tile<tile_index><source_suffix>"
  },
  "splits_written": {
    "train": {
      "split": "train",
      "tiles": 12663,
      "written": 12663,
      "reused": 0,
      "suffixes": {
        ".png": 12663
      }
    },
    "val_source": {
      "split": "val",
      "tiles": 6948,
      "written": 6948,
      "reused": 0,
      "suffixes": {
        ".png": 6948
      }
    }
  },
  "images_written_total": 19611,
  "coco_compatibility": [
    {
      "annotation_file": "train_annotations.json",
      "images": 12663,
      "annotations": 47452,
      "all_images_present": true,
      "all_dimensions_match": tru